In [2]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Python310\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Python310\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "c:\Python310\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "c:\Python310\lib\site-packages\traitlets\config\application.py", line 1041, in launch_instance
    app.start()
  File "c:\Python310\lib\site-packages\ipykernel\kernelapp.py", line 737, in start
    self.i

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [33]:
# Paths
DATA_PATH = os.path.join('..', '..', 'data', 'raw', 'go_emotions_dataset.csv')
OUTPUT_DIR = os.path.join('..', '..', 'data', 'processed')
OUTPUT_FILE = os.path.join(OUTPUT_DIR, 'go_emotions_cleaned.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('DATA_PATH ->', DATA_PATH)
print('OUTPUT_FILE ->', OUTPUT_FILE)

DATA_PATH -> ..\..\data\raw\go_emotions_dataset.csv
OUTPUT_FILE -> ..\..\data\processed\go_emotions_cleaned.csv


In [34]:
df = pd.read_csv(DATA_PATH)
print('Loaded rows, cols:', df.shape)

Loaded rows, cols: (211225, 31)


In [35]:
print('Columns sample:', df.columns[:10].tolist())

Columns sample: ['id', 'text', 'example_very_unclear', 'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion']


In [36]:
df.head(3)

,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,eemcysk,>sexuality shouldn’t be a grouping category I...,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [37]:
# normalize the text (lowercase)
df['clean_text'] = df['text'].astype(str).str.lower()

In [7]:
df.head(3)

,id,text,example_very_unclear,admiration,amusement,anger,annoyance,approval,caring,confusion,...,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral,clean_text
0,eew5j0j,That game hurt.,False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,that game hurt.
1,eemcysk,>sexuality shouldn’t be a grouping category I...,True,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,>sexuality shouldn’t be a grouping category i...
2,ed2mah1,"You do right, if you don't care then fuck 'em!",False,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,"you do right, if you don't care then fuck 'em!"


In [38]:
# Check for urls
url_pattern = r'http\S+|www\S+|https\S+'
urls_exist = df['clean_text'].str.contains(url_pattern, regex=True).sum()
print(f"Rows containing URLs: {urls_exist}")

Rows containing URLs: 16


In [39]:
# Replace urls with [URL]
df['clean_text'] = df['clean_text'].str.replace(url_pattern, '[URL]', regex=True)

In [40]:
url_check = df['clean_text'].str.contains(url_pattern, regex=True)

# Count how many rows still have URLs
print("Rows with URLs remaining:", url_check.sum())

Rows with URLs remaining: 0


In [41]:
# Check for emails
email_pattern = r'\b[\w.-]+@[\w.-]+\.\w{2,4}\b'
emails_exist = df['clean_text'].str.contains(email_pattern, regex=True).sum()
print(f"Rows containing emails: {emails_exist}")

Rows containing emails: 0


In [42]:
# Check for phone numbers
phone_pattern = r'\b\+?\d[\d\s\-()]{6,}\b'
phones_exist = df['clean_text'].str.contains(phone_pattern, regex=True).sum()
print(f"Rows containing phone numbers: {phones_exist}")

Rows containing phone numbers: 114


In [43]:
# Replace phone numbers with [PHONE]
df['clean_text'] = df['clean_text'].str.replace(phone_pattern, '[PHONE]', regex=True)

In [44]:
# Check if any phone numbers still exist in clean_text
phone_remaining = df['clean_text'].str.contains(phone_pattern, regex=True)

# Count how many rows still have phone numbers
print("Rows with phone numbers remaining:", phone_remaining.sum())

Rows with phone numbers remaining: 0


In [47]:
#Check for long numbers (4+ digits)
num_pattern = r'\b\d{4,}\b'
long_nums_exist = df['clean_text'].str.contains(num_pattern, regex=True).sum()
print(f"Rows containing long numbers: {long_nums_exist}")

Rows containing long numbers: 1581


In [48]:
long_nums = df['clean_text'].str.contains(num_pattern, regex=True)
df[long_nums]['clean_text'].sample(10)

120952    they were pretty cool in, like, 2010, but then...
13957     desktop link: ^^/r/helperbot_ ^^downvote ^^to ...
14275     happy new year /r/newyorkmets. everyone in thi...
144249    [name] doesn’t roll his ankle on a 🎾 in 2011 a...
153177    this meme describes how i feel about 2018 pret...
126928    just a quick correction: if the next tax brack...
81180     imagine ordering this. "1000 burgers to the wh...
186110    i would of my school would get me my fucking 1...
158885    ehhh he got played so hard on the [name] vs [n...
142815    deleted ^^^^^^^^^^^^^^^^0.1122 ^^^what ^^^is ^...
Name: clean_text, dtype: object

In [49]:
df['clean_text'] = df['clean_text'].str.replace(num_pattern, '[NUM]', regex=True)

In [50]:
long_nums_remaining = df['clean_text'].str.contains(num_pattern, regex=True)

# Count how many rows still have long numbers
print("Rows with long numbers remaining:", long_nums_remaining.sum())

Rows with long numbers remaining: 0


In [51]:
# Keep letters, numbers, basic punctuation, brackets
df['clean_text'] = df['clean_text'].str.replace(r'[^a-zA-Z0-9\s\[\]\.\,\!\?\-]', '', regex=True)

# Normalize whitespace
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()


In [52]:
texts = df['clean_text'].astype(str)

In [ ]:
mask_map = {
    # ==========================
    # EXISTING STANDARD MASKS
    # ==========================
    r'\[\s*name\s*\]': '[NAME]',
    r'\[\s*religion\s*\]': '[RELIGION]',
    r'\[\s*url\s*\]': '[URL]',
    r'\[\s*user\s*\]': '[USER]',
    
    # ==========================
    # MEDICAL / HEALTH
    # ==========================
    r'\b(doctor|dr|therapist|psychiatrist|psychologist|counselor|psych|psychoanalyst)\b': '[HEALTH_PROF]',
    r'\b(diagnosis|diagnosed|medication|meds|therapy|session|appointment|prescription)\b': '[HEALTH_TERM]',
    r'\b(pill|drug|antidepressant|antipsychotic|benzodiazepine|ssri|snri)\b': '[MED]',
    # Additional conditions / mental health terms
    r'\b(depression|anxiety|panic|bipolar|schizophrenia|ocd|ptsd|adhd)\b': '[CONDITION]',
    r'\b(self[-\s]?harm|suicide|suicidal|cutting|overdose)\b': '[SELF_HARM]',
    r'\b(alcohol|alcoholism|addiction|substance abuse|drugs)\b': '[ADDICTION]',
    
    # ==========================
    # LOCATION
    # ==========================
    r'\b(home|house|apartment|condo|flat|school|college|university|hospital|clinic|work|office|job|store|mall)\s+(of\s+)?mine': '[LOCATION]',
    r'\b(my|our)\s+(city|town|state|neighborhood|street|block)\b': '[LOCATION]',
    # Optional: named locations (using NER if needed)
    
    # ==========================
    # TEMPORAL / DATES / AGE
    # ==========================
    r'\b(?:january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{1,2}(?:st|nd|rd|th)?\b': '[DATE]',
    r'\b(?:\d{1,2})?\s*(?:years?|yrs?|months?|mos?|days?|weeks?)\s+(?:old|ago)\b': '[AGE_TIME]',
    r'\b(19|20)\d{2}\b': '[YEAR]',  # generic years
    
    # ==========================
    # FAMILY / RELATIONSHIPS
    # ==========================
    r'\b(my|our)\s+(mom|dad|mother|father|son|daughter|sister|brother|wife|husband|partner|gf|bf|child|kid|baby)\'s?\b': '[FAMILY]',
    r'\b(my|our)\s+(friend|best friend|buddy|mate)\b': '[FRIEND]',
    
    # ==========================
    # FINANCIAL
    # ==========================
    r'\$\d+(?:,\d{3})*(?:\.\d{2})?': '[MONEY]',
    
    # ==========================
    # ORGANIZATIONS / SCHOOLS
    # ==========================
    r'\b(?:university|college|school|hospital|company|workplace|employer)\s+(?:of|at|in)?\s*mine\b': '[ORG]',
    
    # ==========================
    # LEGAL / SENSITIVE
    # ==========================
    r'\b(court|lawsuit|police|arrest|legal action|judge|trial)\b': '[LEGAL]',
}


In [54]:
# Check which categories exist and how many times
for pattern, token in mask_map.items():
    count = texts.str.contains(pattern, flags=re.IGNORECASE, regex=True).sum()
    if count > 0:
        print(f"{token}: found in {count} rows")

[NAME]: found in 29286 rows
[RELIGION]: found in 457 rows
[URL]: found in 16 rows


C:\Users\PC\AppData\Local\Temp\ipykernel_17964\3657425570.py:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  count = texts.str.contains(pattern, flags=re.IGNORECASE, regex=True).sum()


[HEALTH_PROF]: found in 561 rows
[HEALTH_TERM]: found in 411 rows
[MED]: found in 206 rows
[CONDITION]: found in 638 rows
[SELF_HARM]: found in 110 rows
[ADDICTION]: found in 409 rows
[LOCATION]: found in 5 rows
[LOCATION]: found in 84 rows
[DATE]: found in 79 rows
[AGE_TIME]: found in 1327 rows
[YEAR]: found in 8 rows
[FRIEND]: found in 382 rows
[LEGAL]: found in 708 rows


In [55]:
# Count exact duplicate clean_text rows
duplicate_mask = df['clean_text'].duplicated(keep=False)  # keep=False marks all duplicates as True
num_duplicates = duplicate_mask.sum()
print(f"Number of rows that have exact duplicate clean_text: {num_duplicates}")

Number of rows that have exact duplicate clean_text: 211156


In [56]:
duplicate_examples = df.loc[duplicate_mask, 'clean_text'].value_counts().head(10)
print("\nTop 10 repeated clean_texts:")
print(duplicate_examples)


Top 10 repeated clean_texts:
clean_text
thank you!           67
thank you.           53
happy cake day!      52
thank you            33
youre welcome        29
happy cake day       29
[name]               28
i like it            27
weird flex but ok    27
[name].              25
Name: count, dtype: int64


In [ ]:
df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)
print("Rows after removing exact duplicate clean_text:", df.shape[0])

Rows after removing exact duplicate clean_text: 57580


In [58]:
df['clean_text'] = df['clean_text'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [59]:
null_count = df['clean_text'].isna().sum()
print("Null values in clean_text:", null_count)

Null values in clean_text: 0


In [60]:
# Save dataset
df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved successfully at: {OUTPUT_FILE}")
print("Final dataset shape:", df.shape)

Cleaned dataset saved successfully at: ..\..\data\processed\go_emotions_cleaned.csv
Final dataset shape: (57580, 32)
